# Análise Exploratória de Dados (EDA) — Home Credit Default Risk

**Problema de negócio:** Score de risco de inadimplência na concessão de crédito — prever, no
momento da solicitação, se o cliente ficará inadimplente (`TARGET` = 1).

Este notebook explora os **dados limpos** (`clean_data.parquet`, camada Silver gerada por
`data_sanitization.py`): qualidade da base, valores nulos, distribuição do alvo e comportamento
das principais variáveis. Os insights aqui orientam a construção da ABT (`abt_transform.py`).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
# Dados limpos (camada Silver)
df = pd.read_parquet('../Dados/Silver/clean_data.parquet')
print(f"Shape: {df.shape}")
print(f"Numéricas: {df.select_dtypes('number').shape[1]} | Categóricas: {df.select_dtypes('object').shape[1]}")
df.head()

## 1. Variável alvo (TARGET) — base desbalanceada

In [ ]:
target_counts = df['TARGET'].value_counts().sort_index()
default_rate = df['TARGET'].mean()
print(target_counts)
print(f"Default rate: {default_rate:.2%}")

ax = sns.countplot(x='TARGET', data=df, palette='viridis')
plt.title('Distribuição da TARGET (0 = Adimplente, 1 = Inadimplente)')
total = len(df)
for p in ax.patches:
    ax.annotate(f'{100*p.get_height()/total:.1f}%',
                (p.get_x() + p.get_width()/2, p.get_height()),
                ha='center', va='bottom', size=11)
plt.show()

> 📊 **Como interpretar:** apenas **~8% dos clientes são inadimplentes** — base muito **desbalanceada**.
> Por isso a acurácia enganaria (prever "todo mundo paga" já acertaria 92%); usamos **AUC-ROC, KS e Recall**
> e tratamos o desbalanceamento com `class_weight='balanced'`.

## 2. Qualidade dos dados — valores nulos

In [ ]:
missing = (df.isnull().mean() * 100).sort_values(ascending=False)
missing = missing[missing > 0]
print(f"Colunas com algum nulo: {len(missing)}")
print(f"Colunas com > 50% de nulos: {(missing > 50).sum()}")

top = missing.head(20)
plt.figure(figsize=(10, 7))
sns.barplot(x=top.values, y=top.index, palette='rocket')
plt.title('Top 20 colunas por % de valores nulos')
plt.xlabel('% nulos')
plt.axvline(50, color='red', ls='--', label='limite de descarte (50%)')
plt.legend()
plt.show()

> 📊 **Como interpretar:** cerca de **40 colunas** (bloco de imóvel) têm **>50% de nulos** (à direita da
> linha vermelha) e são **descartadas** na ABT. Exceção: `EXT_SOURCE_1` (~56% nulo) é **mantida** por ser
> um dos preditores mais fortes — o valor preditivo compensa o nulo (imputado depois).

## 3. Idade do cliente vs. inadimplência

In [ ]:
df['AGE_YEARS'] = df['DAYS_BIRTH'] / 365
plt.figure(figsize=(10, 6))
sns.kdeplot(data=df, x='AGE_YEARS', hue='TARGET', common_norm=False, fill=True, palette='coolwarm')
plt.title('Distribuição de idade por TARGET (normalizada)')
plt.xlabel('Idade (anos)')
plt.show()

# Taxa de inadimplência por faixa etária
df['AGE_BIN'] = pd.cut(df['AGE_YEARS'], bins=[20, 30, 40, 50, 60, 70])
age_default = df.groupby('AGE_BIN')['TARGET'].mean() * 100
sns.barplot(x=age_default.index.astype(str), y=age_default.values, palette='coolwarm')
plt.title('Taxa de inadimplência (%) por faixa etária')
plt.ylabel('% inadimplência'); plt.xlabel('Faixa etária')
plt.show()

> 📊 **Como interpretar:** **clientes mais jovens inadimplem mais** — a taxa cai conforme a faixa etária
> sobe (de ~12% nos 20-30 para ~5% acima de 60). Idade é um preditor relevante e intuitivo.

## 4. EXT_SOURCE — scores externos (variáveis mais preditivas)

In [ ]:
ext_cols = [c for c in ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3'] if c in df.columns]
fig, axes = plt.subplots(1, len(ext_cols), figsize=(16, 5))
for ax, col in zip(np.atleast_1d(axes), ext_cols):
    sns.kdeplot(data=df, x=col, hue='TARGET', common_norm=False, fill=True, palette='coolwarm', ax=ax)
    ax.set_title(f'{col} por TARGET')
plt.tight_layout(); plt.show()

print("Correlação dos EXT_SOURCE com TARGET:")
print(df[ext_cols + ['TARGET']].corr()['TARGET'].drop('TARGET'))

> 📊 **Como interpretar:** as curvas de quem paga e quem não paga se **separam visivelmente**, e a
> correlação com `TARGET` é **negativa**: quanto **maior** o score externo, **menor** o risco. São as
> **variáveis mais preditivas** da base — confirmado depois pela importância de features do modelo.

## 5. Renda, crédito e comprometimento

In [ ]:
fin_cols = ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY']
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, col in zip(axes, fin_cols):
    sns.boxplot(data=df, x='TARGET', y=col, palette='viridis', ax=ax, showfliers=False)
    ax.set_yscale('log')
    ax.set_title(f'{col} por TARGET (escala log)')
plt.tight_layout(); plt.show()

> 📊 **Como interpretar:** em escala log, as distribuições de renda/crédito de adimplentes e inadimplentes
> são **parecidas** — renda sozinha separa pouco. A forte **assimetria/outliers** motiva as features de
> **razão** na ABT (`CREDIT_INCOME_RATIO`, `ANNUITY_INCOME_RATIO`), que capturam o *comprometimento* melhor
> que os valores absolutos.

## 6. Inadimplência por variáveis categóricas

In [ ]:
cat_cols = [c for c in ['CODE_GENDER', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_INCOME_TYPE'] if c in df.columns]
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
for ax, col in zip(axes.ravel(), cat_cols):
    rate = (df.groupby(col)['TARGET'].mean() * 100).sort_values(ascending=False)
    sns.barplot(x=rate.values, y=rate.index, palette='rocket', ax=ax)
    ax.axvline(df['TARGET'].mean()*100, color='blue', ls='--')
    ax.set_title(f'% inadimplência por {col}')
    ax.set_xlabel('% inadimplência')
plt.tight_layout(); plt.show()

> 📊 **Como interpretar:** a linha azul é a **média geral** (~8%). Categorias **à direita dela** têm risco
> **acima da média** — ex.: menor escolaridade e alguns tipos de renda inadimplem mais. As diferenças são
> moderadas, mas ainda assim úteis ao modelo.

## 7. Correlação das principais numéricas com o alvo

In [ ]:
cols = [c for c in ['TARGET', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3',
                   'AGE_YEARS', 'DAYS_EMPLOYED', 'AMT_INCOME_TOTAL', 'AMT_CREDIT',
                   'AMT_ANNUITY', 'AMT_GOODS_PRICE'] if c in df.columns]
corr = df[cols].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', vmin=-1, vmax=1)
plt.title('Matriz de correlação')
plt.show()

> 📊 **Como interpretar:** os **`EXT_SOURCE_*`** são os mais (negativamente) correlacionados com o alvo.
> Renda/crédito têm correlação **fraca** com `TARGET` diretamente, mas correlação **alta entre si**
> (crédito × valor do bem) — sinal de redundância, esperado nessas variáveis financeiras.
>
> **Por que as variáveis de correlação *negativa* têm a relação mais forte?** O **sinal** só indica a
> *direção* da relação (negativo = "quanto **maior** a variável, **menor** o risco de calote"); quem
> mede a *força* é o **módulo** do valor (perto de 0 = fraca; perto de ±1 = forte). Acontece que, nesta
> base, as variáveis com maior módulo são justamente as **negativas** — e elas são **fatores de
> proteção/estabilidade**:
> - **`EXT_SOURCE_*`** são scores de crédito já prontos de outras fontes: condensam num único número o
>   histórico financeiro do cliente, então naturalmente concentram o sinal (score externo alto → risco baixo).
> - **`AGE_YEARS`** (idade) e **`DAYS_EMPLOYED`** (tempo de emprego) capturam **maturidade e estabilidade
>   financeira**: mais idade / mais tempo empregado → menor inadimplência.
>
> Já os valores financeiros **absolutos** (renda, crédito) têm correlação **fraca** porque o que importa
> para o risco não é o valor em si, mas o **comprometimento relativo** (parcela ÷ renda) — exatamente o
> que as features de **razão** da ABT capturam. Ou seja: não é o sinal negativo que "causa" a força; é que
> os melhores indicadores de solvência desta base agem **reduzindo** o risco.

## 8. Força de associação com o alvo — Eta² (numéricas) e Cramér's V (categóricas)

A correlação de Pearson (item 7) só mede relação **linear** e **só serve para numéricas**. Para
ranquear o poder de cada variável de forma comparável, usamos duas medidas de **força de associação**
(ambas variam de **0 = nenhuma associação** a **1 = associação total**):

- **Eta² (η²)** para **variáveis numéricas**: fração da variância da variável que é explicada pela
  separação entre adimplentes e inadimplentes. Capta relação **não-linear** (ao contrário de Pearson).
- **Cramér's V** para **variáveis categóricas**: força da associação entre a categoria e a `TARGET`,
  derivada do teste qui-quadrado. É o equivalente "categórico" da correlação.

In [ ]:
from scipy.stats import chi2_contingency

def eta_squared(num, cat):
    # Fração da variância de `num` explicada pelos grupos de `cat` (aqui, TARGET 0/1).
    d = pd.DataFrame({'num': num, 'cat': cat}).dropna()
    grand_mean = d['num'].mean()
    ss_total = ((d['num'] - grand_mean) ** 2).sum()
    ss_between = d.groupby('cat')['num'].apply(lambda g: len(g) * (g.mean() - grand_mean) ** 2).sum()
    return ss_between / ss_total if ss_total else 0.0

def cramers_v(x, y):
    # Força da associação entre duas categóricas, a partir do qui-quadrado (0 a 1).
    confusion = pd.crosstab(x, y)
    chi2 = chi2_contingency(confusion)[0]
    n = confusion.to_numpy().sum()
    r, k = confusion.shape
    return np.sqrt(chi2 / (n * (min(r, k) - 1))) if min(r, k) > 1 else 0.0

num_cols = [c for c in ['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'AGE_YEARS', 'DAYS_EMPLOYED',
                        'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE'] if c in df.columns]
cat_cols = [c for c in ['CODE_GENDER', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_INCOME_TYPE',
                        'NAME_CONTRACT_TYPE', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY'] if c in df.columns]

eta = pd.Series({c: eta_squared(df[c], df['TARGET']) for c in num_cols}).sort_values(ascending=False)
cramer = pd.Series({c: cramers_v(df[c], df['TARGET']) for c in cat_cols}).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.barplot(x=eta.values, y=eta.index, palette='mako', ax=axes[0])
axes[0].set_title('Eta² (η²) — numéricas vs. TARGET'); axes[0].set_xlabel('η² (0 a 1)')
sns.barplot(x=cramer.values, y=cramer.index, palette='flare', ax=axes[1])
axes[1].set_title("Cramér's V — categóricas vs. TARGET"); axes[1].set_xlabel("Cramér's V (0 a 1)")
plt.tight_layout(); plt.show()

print('Eta² (numéricas):'); print(eta.round(4))
print('\nCramér\'s V (categóricas):'); print(cramer.round(4))

> 📊 **Como interpretar:** em ambas, **mais alto = associação mais forte com o alvo** (escala 0 a 1),
> e o sinal não importa (são sempre positivas). Regra prática de leitura: **< 0,01** associação
> desprezível, **~0,01–0,06** fraca, **~0,06–0,14** moderada, **> 0,14** forte.
>
> - **Eta² (η²):** os **`EXT_SOURCE_*`** lideram de novo — agora medindo a relação de forma
>   **não-linear** (e ainda assim dominam), confirmando que são os preditores mais fortes. Valores
>   financeiros absolutos (renda, crédito) ficam com η² **baixo**: separam pouco quem paga de quem não paga.
> - **Cramér's V:** entre as categóricas, **escolaridade** e **tipo de renda** costumam ter a maior
>   associação, mas em geral os valores são **baixos** (associação fraca a moderada) — as categóricas
>   ajudam pouco isoladamente, mas somam ao modelo. Atenção: com a base grande, o qui-quadrado dá
>   "significância" a quase tudo; por isso olhamos o **tamanho do Cramér's V**, não só o p-valor.

## 9. `clean_bureau` — histórico de crédito externo (nível de registro original)

As outras **duas** tabelas da Silver (`clean_bureau` e `clean_previous_application`) trazem o **histórico
de crédito** do cliente, cada uma no seu **nível de registro original** — antes de qualquer agregação.

> ⚠️ Elas **não têm `TARGET`** (cada registro é um crédito/pedido, não um cliente), então aqui a
> exploração é **descritiva**: entender volume, cardinalidade e composição. A relação com a inadimplência
> só aparece **depois de agregar** por `SK_ID_CURR` (em `feature_aggregation.py`), o que é explorado no
> `abt_overview.ipynb`. O ponto-chave: **vários registros por cliente** — é o que exige agregar antes do join.

Começando pelo **`clean_bureau`** = crédito em **outras instituições**, reportado ao birô de crédito
(tipo Serasa/SPC). 1 linha por crédito (`SK_ID_BUREAU`):

In [ ]:
bureau = pd.read_parquet('../Dados/Silver/clean_bureau.parquet')
print(f"Shape: {bureau.shape[0]:,} × {bureau.shape[1]} | clientes: {bureau['SK_ID_CURR'].nunique():,} | nível de registro: 1 linha por SK_ID_BUREAU")

b_per = bureau.groupby('SK_ID_CURR').size()                          # créditos por cliente
active = bureau['CREDIT_ACTIVE'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
# clip(upper=20): quem tem 20+ créditos vira uma única barra "20+" (a cauda real vai até o máx);
# é só p/ leitura — não altera a feature BUREAU_CREDIT_COUNT, que guarda o valor real.
sns.histplot(b_per.clip(upper=20), bins=20, ax=axes[0], color='#4c72b0')
axes[0].set_title(f'Créditos externos por cliente (mediana {int(b_per.median())}, máx {b_per.max()})')
axes[0].set_xlabel('nº de créditos no bureau'); axes[0].set_ylabel('clientes')
axes[0].set_xticks([0, 5, 10, 15, 20]); axes[0].set_xticklabels(['0', '5', '10', '15', '20+'])
sns.barplot(x=active.values, y=active.index, palette='viridis', ax=axes[1])
axes[1].set_title('Status dos créditos (CREDIT_ACTIVE)')
axes[1].set_xlabel('% dos créditos')
plt.tight_layout(); plt.show()

> 📊 **Como interpretar:** cada cliente tem **vários** créditos externos (mediana **4**, chegando a 116) —
> por isso a tabela **não entra direto** na ABT (1 linha/cliente): é preciso **agregar** por `SK_ID_CURR`
> (contagem, soma da dívida, atrasos → features `BUREAU_*`), transformando N linhas em uma. Na composição,
> a maioria dos créditos já está **encerrada** (~63% `Closed`) e ~37% seguem **ativos** — o volume de
> crédito ativo e o histórico de atraso são justamente o sinal de risco que a agregação captura. Cliente
> sem histórico não aparece aqui e recebe **0** nas features agregadas ("sem histórico").
>
> *(No histograma, a última barra **"20+"** agrupa quem tem 20 ou mais créditos — a cauda real segue até
> 116, mas com pouquíssimos clientes; é só recorte visual e não afeta a feature `BUREAU_CREDIT_COUNT`.)*

## 10. `clean_previous_application` — histórico de crédito interno (nível de registro original)

A segunda tabela auxiliar: **pedidos de crédito anteriores do cliente na própria Home Credit**
(aprovados, recusados, cancelados). 1 linha por pedido (`SK_ID_PREV`); vários por cliente. É um histórico
**independente** do bureau — ambos se ligam ao cliente por `SK_ID_CURR`, mas não entre si.

In [ ]:
prev = pd.read_parquet('../Dados/Silver/clean_previous_application.parquet')
print(f"Shape: {prev.shape[0]:,} × {prev.shape[1]} | clientes: {prev['SK_ID_CURR'].nunique():,} | nível de registro: 1 linha por SK_ID_PREV")

p_per = prev.groupby('SK_ID_CURR').size()                             # pedidos por cliente
status = prev['NAME_CONTRACT_STATUS'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
# clip(upper=20): quem tem 20+ pedidos vira uma única barra "20+" (a cauda real vai até o máx);
# só p/ leitura — a feature PREV_APP_COUNT guarda o valor real.
sns.histplot(p_per.clip(upper=20), bins=20, ax=axes[0], color='#55a868')
axes[0].set_title(f'Pedidos anteriores por cliente (mediana {int(p_per.median())}, máx {p_per.max()})')
axes[0].set_xlabel('nº de pedidos na Home Credit'); axes[0].set_ylabel('clientes')
axes[0].set_xticks([0, 5, 10, 15, 20]); axes[0].set_xticklabels(['0', '5', '10', '15', '20+'])
sns.barplot(x=status.values, y=status.index, palette='rocket', ax=axes[1])
axes[1].set_title('Resultado do pedido (NAME_CONTRACT_STATUS)')
axes[1].set_xlabel('% dos pedidos')
plt.tight_layout(); plt.show()

> 📊 **Como interpretar:** mesma lógica do bureau — **vários pedidos por cliente** (mediana **4**, até 77),
> o que exige **agregar** por `SK_ID_CURR` (→ features `PREV_*`). Na composição, ~62% dos pedidos foram
> **aprovados**, mas há fatias relevantes de **recusados** (~17%) e **cancelados** (~19%): a **taxa de
> recusa** histórica (`PREV_REFUSED_RATE`) é um sinal de risco intuitivo — quem já foi recusado antes
> tende a ser mais arriscado. Como no bureau, a relação com o alvo aparece só após a agregação (ver `abt_overview.ipynb`).
>
> *(A última barra **"20+"** do histograma agrupa quem tem 20 ou mais pedidos — a cauda real vai até 77;
> só recorte visual, `PREV_APP_COUNT` guarda o valor real. Obs.: hoje as somas de valor agregam **todos**
> os status — inclusive recusados/cancelados; o status entra como contagem/taxa, não como filtro.)*

## Principais conclusões da EDA
- **Base desbalanceada** (~8% de inadimplência) → exige tratamento (`class_weight='balanced'`) e
  métricas adequadas (AUC-ROC, KS, Recall da classe 1) em vez de acurácia.
- **Muitos nulos**: ~40 colunas (bloco de imóvel) têm >50% de nulos e serão descartadas; porém
  `EXT_SOURCE_1` (56% nulo) é preservada por seu poder preditivo.
- **EXT_SOURCE_1/2/3** são as variáveis mais correlacionadas com o alvo (correlação negativa:
  quanto maior o score externo, menor o risco).
- **Idade**: clientes mais jovens inadimplem mais.
- Renda/crédito têm forte assimetria (outliers) → motivam as features de **razão** na ABT
  (`CREDIT_INCOME_RATIO`, `ANNUITY_INCOME_RATIO`, `ANNUITY_CREDIT_RATIO`).
- **Força de associação** (η² para numéricas, Cramér's V para categóricas) confirma o ranking: os
  `EXT_SOURCE_*` dominam mesmo medindo relação **não-linear**; as categóricas têm associação fraca a
  moderada (escolaridade/tipo de renda à frente).
- **Históricos de crédito (auxiliares da Silver)**: `bureau` (externo) e `previous_application` (interno)
  têm **vários registros por cliente** (mediana 4 cada) e **sem `TARGET`** no nível de registro original →
  precisam ser **agregados** por `SK_ID_CURR` (features `BUREAU_*`/`PREV_*`) antes de entrar na ABT; a
  relação com o alvo é explorada no `abt_overview.ipynb`.